# tinyLMTune — Text Summarization

This notebook demonstrates 3 ways to train TinyBERT for **summarization** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (xsum)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
# !pip install -e .

from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation, plot_results

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [ ]:
best = optimize_slm(
    task="summarization",
    corpus_prompt="Generate news articles with summaries about technology and science",
    n_examples=1000,
    pop_size=4,
    generations=2,
    max_len=192,
    output_dir="models/summarization_synthetic",
)
print("Best config:", best)

### Model Inference (Synthetic Data)

In [ ]:
model = TinyInference("models/summarization_synthetic")
result = model.predict(
    "The European Space Agency announced today that its new Mars rover "
    "has successfully landed on the surface of Mars. The rover, named "
    "Athena, will spend the next two years collecting soil samples and "
    "analyzing the atmosphere for signs of past microbial life."
)
print("Output:", result)

---
## Example 2 — Benchmark Data (xsum)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load xsum dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("xsum", split="train")
ds = ds.shuffle(seed=42).select(range(1000))

benchmark_data = [
    {"text": r["document"][:1000], "summary": r["summary"]}
    for r in ds
]

print(f"Loaded {len(benchmark_data)} records")
print(f"Text preview: {benchmark_data[0]['text'][:100]}...")
print(f"Summary: {benchmark_data[0]['summary']}")

### Analyze token lengths

In [ ]:
print_token_analysis(benchmark_data, task="summarization")

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="summarization")

### Train with GA optimisation

In [ ]:
best = optimize_slm(
    task="summarization",
    user_data=benchmark_data,
    max_len=256,
    pop_size=4,
    generations=2,
    output_dir="models/summarization_benchmark",
)
print("Best config:", best)

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/summarization_benchmark")
result = model.predict("Scientists have discovered a new species of deep-sea fish.")
print("Output:", result)

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"text": "The company reported record quarterly earnings of $5 billion, driven by strong growth in its cloud computing division. CEO Jane Smith attributed the success to increased enterprise adoption and new AI-powered services launched in Q3.", "summary": "Company posts record $5B quarterly earnings from cloud growth."},
    {"text": "A magnitude 6.2 earthquake struck off the coast of Japan early Monday morning, triggering tsunami warnings across the Pacific. Authorities evacuated coastal communities and emergency services are assessing the damage.", "summary": "6.2 earthquake near Japan triggers tsunami warnings and evacuations."},
    {"text": "Researchers at MIT have developed a new type of battery that can charge in under five minutes and last for over 1000 cycles. The technology uses a novel lithium-iron-phosphate chemistry.", "summary": "MIT creates fast-charging battery lasting 1000+ cycles."},
    {"text": "The World Health Organization declared the mpox outbreak a global health emergency, urging countries to increase vaccination efforts and improve surveillance systems.", "summary": "WHO declares mpox a global health emergency."},
    {"text": "SpaceX successfully launched its 50th Starlink mission of the year, deploying 60 satellites into low Earth orbit to expand global internet coverage.", "summary": "SpaceX completes 50th Starlink launch with 60 satellites."},
    {"text": "New York City announced plans to ban gas-powered vehicles from Manhattan by 2035, making it the first major US city to implement such a policy.", "summary": "NYC to ban gas vehicles in Manhattan by 2035."},
    {"text": "Apple unveiled its latest mixed reality headset at WWDC, featuring eye tracking, spatial audio, and a new operating system designed for immersive computing.", "summary": "Apple launches mixed reality headset with eye tracking."},
    {"text": "A study published in Nature found that urban green spaces reduce mental health issues by up to 30 percent among city residents who visit them weekly.", "summary": "Urban parks cut mental health issues by 30% with weekly visits."},
    {"text": "The Federal Reserve held interest rates steady at 5.25 percent, signaling a wait-and-see approach amid mixed economic indicators and cooling inflation.", "summary": "Fed holds rates at 5.25% amid mixed economic signals."},
    {"text": "Formula One announced Las Vegas as a permanent fixture on its calendar, signing a 10-year deal to host a night race on the famous Strip.", "summary": "F1 signs 10-year Las Vegas night race deal."},
]

best = optimize_slm(
    task="summarization",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/summarization_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw text block — split into paragraphs, structured via Flan-T5
raw_text = """
The global semiconductor shortage continued to impact automotive production 
in the third quarter. Major manufacturers reported delays of up to six months 
for new vehicle deliveries.

Meanwhile, chip makers are investing heavily in new fabrication facilities. 
TSMC announced a $40 billion expansion in Arizona, while Intel is building 
two new plants in Ohio.

Industry analysts predict the shortage will ease by mid-2025 as new capacity 
comes online. However, demand for advanced chips in AI applications may 
offset the additional supply.
"""

best = optimize_slm(
    task="summarization",
    user_data=raw_text,
    split_strategy="paragraph",
    pop_size=4,
    generations=1,
    output_dir="models/summarization_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with non-standard keys
wrong_format = [
    {"article": "Global temperatures reached a new record high in July, with average readings 1.5 degrees above pre-industrial levels.", "brief": "July breaks global temperature records."},
    {"article": "The tech giant announced layoffs affecting 10,000 employees as part of a restructuring to focus on AI development.", "brief": "Tech company cuts 10K jobs to pivot to AI."},
]

# Pipeline detects these don't match {"text", "summary"}, extracts text, structures via Flan-T5
best = optimize_slm(
    task="summarization",
    user_data=wrong_format,
    pop_size=4,
    generations=1,
    output_dir="models/summarization_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/summarization_user")
result = model.predict("The central bank raised interest rates by 25 basis points.")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | xsum | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).